In [6]:
!pip install groq sentence-transformers faiss-cpu PyPDF2

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq
import PyPDF2
import time
import textwrap

# Your Groq API key
API_KEY = "paste_your_api_key"
client = Groq(api_key=API_KEY)

# Load embedding model
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Setup Complete! RAG System Ready!")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Setup Complete! RAG System Ready!


In [7]:
# DOCUMENT PROCESSING MODULE

def load_text_document(text):
    """Load from raw text"""
    return text

def chunk_document(text, chunk_size=200, overlap=50):
    """Split document into overlapping chunks"""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# Sample research document
sample_document = """
Artificial Intelligence and Machine Learning have transformed
modern technology. Deep learning models use neural networks
with multiple layers to learn complex patterns from data.

Natural Language Processing enables computers to understand
human language. Transformer architecture revolutionized NLP
by introducing attention mechanisms. BERT and GPT are popular
transformer based models used in many applications.

Large Language Models are trained on massive text datasets.
They can generate human like text, answer questions, summarize
documents, and perform many language tasks. However they have
limitations like hallucination where they generate false information.

Retrieval Augmented Generation combines information retrieval
with text generation. It retrieves relevant documents and uses
them as context for the language model. This reduces hallucination
and improves factual accuracy significantly.

Computer Vision enables machines to interpret visual information.
Convolutional Neural Networks are widely used for image classification.
Object detection models like YOLO can identify multiple objects in images.

Reinforcement Learning trains agents through reward and punishment.
AlphaGo used reinforcement learning to defeat world champions in Go.
Modern robots use reinforcement learning for complex motor tasks.

Multimodal AI combines multiple data types like text, images and audio.
These systems can understand and generate content across modalities.
Applications include image captioning, visual question answering and more.
"""

# Process document
print("Processing document...")
chunks = chunk_document(sample_document)
print(f"Document split into {len(chunks)} chunks")

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(chunk[:100] + "...")

print("\nDocument Processing Complete!")

Processing document...
Document split into 2 chunks

Chunk 1:
Artificial Intelligence and Machine Learning have transformed modern technology. Deep learning model...

Chunk 2:
through reward and punishment. AlphaGo used reinforcement learning to defeat world champions in Go. ...

Document Processing Complete!


In [8]:
# VECTOR DATABASE MODULE

print("Building Vector Database...")
print("=" * 60)

# Create embeddings for all chunks
chunk_embeddings = embedder.encode(chunks)
print(f"Created {len(chunk_embeddings)} embeddings")
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")

# Build FAISS index
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype('float32'))
print(f"Vector database built with {index.ntotal} vectors")

def retrieve_relevant_chunks(question, top_k=3):
    """Find most relevant chunks for a question"""
    question_embedding = embedder.encode([question])
    distances, indices = index.search(
        question_embedding.astype('float32'), top_k
    )
    relevant_chunks = [chunks[i] for i in indices[0]]
    return relevant_chunks

# Test retrieval
test_question = "What is RAG and how does it work?"
relevant = retrieve_relevant_chunks(test_question)
print(f"\nTest Question: {test_question}")
print(f"\nTop 3 Relevant Chunks Retrieved:")
for i, chunk in enumerate(relevant):
    print(f"\nChunk {i+1}: {chunk[:150]}...")

print("\nVector Database Module Complete!")

Building Vector Database...
Created 2 embeddings
Embedding dimension: 384
Vector database built with 2 vectors

Test Question: What is RAG and how does it work?

Top 3 Relevant Chunks Retrieved:

Chunk 1: Artificial Intelligence and Machine Learning have transformed modern technology. Deep learning models use neural networks with multiple layers to lear...

Chunk 2: through reward and punishment. AlphaGo used reinforcement learning to defeat world champions in Go. Modern robots use reinforcement learning for compl...

Chunk 3: through reward and punishment. AlphaGo used reinforcement learning to defeat world champions in Go. Modern robots use reinforcement learning for compl...

Vector Database Module Complete!


In [9]:
# RAG QUESTION ANSWERING MODULE

print("RAG QA System")
print("=" * 60)

def rag_answer(question):
    """Complete RAG pipeline"""

    print(f"\nQuestion: {question}")
    print("-" * 40)

    # Step 1 Retrieve relevant chunks
    relevant_chunks = retrieve_relevant_chunks(question, top_k=3)
    context = "\n\n".join(relevant_chunks)

    # Step 2 Build prompt with context
    prompt = f"""You are a helpful research assistant.
Answer the question based ONLY on the context provided below.
If the answer is not in the context, say "I cannot find this in the document."

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    # Step 3 Generate answer
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )

    answer = response.choices[0].message.content
    print(f"Answer: {answer}")
    print("-" * 40)
    time.sleep(2)
    return answer

# Test with multiple questions
questions = [
    "What is Retrieval Augmented Generation?",
    "What are the limitations of Large Language Models?",
    "How does Computer Vision work?",
    "What is reinforcement learning used for?",
    "What is multimodal AI?"
]

answers = []
for question in questions:
    answer = rag_answer(question)
    answers.append(answer)

print("\nRAG QA Module Complete!")

RAG QA System

Question: What is Retrieval Augmented Generation?
----------------------------------------
Answer: Retrieval Augmented Generation combines information retrieval with text generation. It retrieves relevant documents and uses them as context for the language model. This reduces hallucination and improves factual accuracy significantly.
----------------------------------------

Question: What are the limitations of Large Language Models?
----------------------------------------
Answer: hallucination where they generate false information.
----------------------------------------

Question: How does Computer Vision work?
----------------------------------------
Answer: Computer Vision enables machines to interpret visual information. Convolutional Neural Networks are widely used for image classification. Object detection models like YOLO can identify multiple objects in images.
----------------------------------------

Question: What is reinforcement learning used for?
------

In [10]:
# COMPARISON: RAG vs WITHOUT RAG

print("COMPARISON: RAG vs WITHOUT RAG")
print("=" * 60)

def answer_without_rag(question):
    """Answer without any context"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": question}],
        temperature=0.1
    )
    return response.choices[0].message.content

test_q = "What are the limitations of Large Language Models?"

print(f"\nQuestion: {test_q}")
print("\n WITHOUT RAG (No Context):")
print("-" * 40)
without_rag = answer_without_rag(test_q)
print(without_rag)
time.sleep(3)

print("\n WITH RAG (With Retrieved Context):")
print("-" * 40)
with_rag = rag_answer(test_q)

print("\nKey Difference:")
print("WITHOUT RAG: General knowledge, possible hallucination")
print("WITH RAG: Grounded in document, more accurate")
print("\nComparison Complete!")

COMPARISON: RAG vs WITHOUT RAG

Question: What are the limitations of Large Language Models?

 WITHOUT RAG (No Context):
----------------------------------------
Large Language Models (LLMs) have made significant progress in natural language processing, but they also have several limitations. Some of the key limitations of LLMs include:

1. **Lack of Common Sense**: While LLMs can process and generate human-like text, they often lack common sense and real-world experience. They may not understand the nuances of human behavior, social norms, or cultural context.
2. **Limited Domain Knowledge**: LLMs are typically trained on a specific dataset and may not have in-depth knowledge of a particular domain or subject area. They may struggle to provide accurate information or answer complex questions outside their training data.
3. **Biased and Sensitive Information**: LLMs can perpetuate biases and stereotypes present in their training data, which can lead to discriminatory or insensitive res

In [11]:
# FINAL SUMMARY

print("RAG SYSTEM — PROJECT SUMMARY")
print("=" * 60)
print("""
SYSTEM COMPONENTS:
1. Document Processor  — Chunks text into overlapping segments
2. Embedding Engine    — Converts text to vectors using MiniLM
3. Vector Database     — FAISS index for fast similarity search
4. RAG Pipeline        — Retrieves context + generates answers
5. Comparison Module   — RAG vs No RAG analysis

TECH STACK:
- Python
- Groq API + LLaMA 3.1
- Sentence Transformers
- FAISS Vector Database
- PyPDF2

KEY FINDINGS:
1. RAG significantly reduces hallucination
2. Context retrieval improves factual accuracy
3. Chunking strategy affects retrieval quality
4. Overlap between chunks preserves context

RESEARCH RELEVANCE:
- Grounded text generation
- LLM hallucination mitigation
- Information retrieval
- Knowledge base QA systems
""")
print("PROJECT COMPLETE!")
print("RAG Document QA System — Sneha Kumari J")

RAG SYSTEM — PROJECT SUMMARY

SYSTEM COMPONENTS:
1. Document Processor  — Chunks text into overlapping segments
2. Embedding Engine    — Converts text to vectors using MiniLM
3. Vector Database     — FAISS index for fast similarity search
4. RAG Pipeline        — Retrieves context + generates answers
5. Comparison Module   — RAG vs No RAG analysis

TECH STACK:
- Python
- Groq API + LLaMA 3.1
- Sentence Transformers
- FAISS Vector Database
- PyPDF2

KEY FINDINGS:
1. RAG significantly reduces hallucination
2. Context retrieval improves factual accuracy
3. Chunking strategy affects retrieval quality
4. Overlap between chunks preserves context

RESEARCH RELEVANCE:
- Grounded text generation
- LLM hallucination mitigation
- Information retrieval
- Knowledge base QA systems

PROJECT COMPLETE!
RAG Document QA System — Sneha Kumari J
